# Cleaning V3 Sample 1000 Smoke Validation

This Notebook verifies the refactored cleaning planner, scheduler, evaluator, state, and export flow on the default MinIO `sample_1000` dataset.

## 0. Bootstrap repo root

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if not (repo_root / "pyproject.toml").exists():
    raise RuntimeError("cannot locate repository root")

src_root = repo_root / "src"
for path in [repo_root, src_root]:
    path_text = str(path)
    if path_text in sys.path:
        sys.path.remove(path_text)
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(src_root))

print("repo_root:", repo_root)
print("src_root:", src_root)


## 1. Imports and runtime paths

In [ ]:
import json

import pandas as pd

from image_gallery.cleaning import BasicCleaner
from notebooks._helpers.cleaning_configs import get_cleaning_v3_first_batch_operator_configs
from notebooks._helpers.datasets import (
    get_default_minio_sample_1000_raw_path,
    load_default_minio_sample_1000_dataset,
    load_default_minio_sample_1000_frame,
)
from notebooks._helpers.paths import get_notebook_library_root, reset_output_dir

pd.set_option("display.max_columns", 80)

RUN_ROOT = get_notebook_library_root("cleaning_v3_sample_1000")
RUN_OUTPUT_DIR = reset_output_dir(RUN_ROOT / "run")
EXPORT_DIR = reset_output_dir(RUN_ROOT / "exports")
PREVIEW_DIR = reset_output_dir(RUN_ROOT / "preview")

raw_path = get_default_minio_sample_1000_raw_path()
raw_frame = load_default_minio_sample_1000_frame()

print("raw_path:", raw_path)
print("run_output_dir:", RUN_OUTPUT_DIR)
print("export_dir:", EXPORT_DIR)
print("preview_dir:", PREVIEW_DIR)


## 2. Validate raw dataset

In [ ]:
required_raw_columns = {"image_id", "image_uri"}
missing_raw_columns = sorted(required_raw_columns.difference(raw_frame.columns))
if missing_raw_columns:
    raise AssertionError(f"missing raw dataset columns: {missing_raw_columns}")
if raw_frame["image_uri"].isna().any():
    raise AssertionError("raw dataset contains empty image_uri values")

print("raw_rows:", len(raw_frame))
print("raw_columns:", raw_frame.columns.tolist())
raw_frame[["image_id", "image_uri"]].head()


## 3. Load storage-backed dataset

In [ ]:
dataset = load_default_minio_sample_1000_dataset()

sample_images = []
for image_uri in raw_frame["image_uri"].astype(str).head(3):
    image = dataset.read_image(image_uri)
    sample_images.append(
        {
            "image_uri": image_uri,
            "size": image.size,
            "format": image.format,
        }
    )

pd.DataFrame(sample_images)


## 4. Compile and inspect execution plan

In [ ]:
operator_configs = get_cleaning_v3_first_batch_operator_configs()
operator_names = [next(iter(item.keys())) for item in operator_configs]

cleaner = BasicCleaner(operator_configs, output_dir=RUN_OUTPUT_DIR)
plan_frame = cleaner.plan()

expected_plan_columns = {
    "step_index",
    "computer_name",
    "execution_mode",
    "requested_parameters",
    "required_parameters",
    "produced_parameters",
    "upstream_computers",
}
missing_plan_columns = sorted(expected_plan_columns.difference(plan_frame.columns))
if missing_plan_columns:
    raise AssertionError(f"missing plan columns: {missing_plan_columns}")
if plan_frame.empty:
    raise AssertionError("compiled parameter plan is empty")

execution_modes = set(plan_frame["execution_mode"].astype(str))
required_modes = {"per_image", "dataset_aggregate"}
missing_modes = sorted(required_modes.difference(execution_modes))
if missing_modes:
    raise AssertionError(f"missing execution modes: {missing_modes}")

print("operator_names:", operator_names)
print("execution_modes:", sorted(execution_modes))
plan_frame


## 5. Run BasicCleaner

In [ ]:
dataset = load_default_minio_sample_1000_dataset()
result = cleaner.run(dataset, progress="auto")
preview = result.preview(limit=5)
state_frame = result.state()
preview_html_path = result.preview_html(PREVIEW_DIR / "quality_blur.html", operator_name="blur")

print("preview:", preview)
state_frame


## 6. Validate exported result views


In [ ]:
parameter_table_path = result.export_table("parameter", RUN_OUTPUT_DIR / "parameter_table.parquet")
evaluation_table_path = result.export_table("evaluation", RUN_OUTPUT_DIR / "evaluation_table.parquet")
manifest_path = result.export_manifest(RUN_OUTPUT_DIR / "parameter_manifest.json")

for output_path in [parameter_table_path, evaluation_table_path, manifest_path, preview_html_path]:
    if not output_path.exists():
        raise AssertionError(f"missing output file: {output_path}")

parameter_table = pd.read_parquet(parameter_table_path)
evaluation_table = pd.read_parquet(evaluation_table_path)
manifest_payload = json.loads(manifest_path.read_text(encoding="utf-8"))
parameter_manifest = manifest_payload["parameter_manifest"]
operator_outputs = manifest_payload["operator_outputs"]

required_parameter_columns = {
    "decode_ok",
    "decode_error",
    "width",
    "height",
    "aspect_ratio",
    "megapixels",
    "blur_score",
    "brightness_score",
    "contrast_score",
    "blank_score",
    "content_hash",
    "exact_duplicate_group_id",
    "exact_duplicate_count",
}
missing_parameter_columns = sorted(required_parameter_columns.difference(parameter_table.columns))
if missing_parameter_columns:
    raise AssertionError(f"missing parameter columns: {missing_parameter_columns}")

required_evaluation_columns = {"image_id", "image_uri", "final_action", "final_reason"}
missing_evaluation_columns = sorted(required_evaluation_columns.difference(evaluation_table.columns))
if missing_evaluation_columns:
    raise AssertionError(f"missing evaluation columns: {missing_evaluation_columns}")

if len(parameter_table) != len(raw_frame):
    raise AssertionError("parameter table row count does not match raw frame")
if len(evaluation_table) != len(raw_frame):
    raise AssertionError("evaluation table row count does not match raw frame")
if result.status() != "completed":
    raise AssertionError(f"unexpected cleaner state: {result.status()}")

print("parameter_table_shape:", parameter_table.shape)
print("evaluation_table_shape:", evaluation_table.shape)
print("preview_html_path:", preview_html_path)
print("manifest_parameters:", sorted(parameter_manifest))
print("operator_outputs:", sorted(operator_outputs))


## 7. Summary checks

In [ ]:
action_counts = (
    evaluation_table["final_action"]
    .value_counts(dropna=False)
    .rename_axis("final_action")
    .reset_index(name="count")
)

operator_status_counts = (
    state_frame["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="count")
)

display(action_counts)
display(operator_status_counts)
state_frame


## 8. Inspect duplicate or dropped rows

In [ ]:
analysis_columns = [
    "image_id",
    "image_uri",
    "final_action",
    "final_reason",
    "exact_duplicate_action",
    "exact_duplicate_reason",
    "exact_duplicate_group_id",
    "exact_duplicate_count",
]
analysis_columns = [column for column in analysis_columns if column in evaluation_table.columns]
analysis_frame = evaluation_table[analysis_columns].copy()

dropped_or_duplicate = analysis_frame.query(
    "final_action != 'keep' or exact_duplicate_count > 1",
    engine="python",
).head(10)

dropped_or_duplicate


## 9. Validate exports

In [ ]:
full_export = result.export("full", str(EXPORT_DIR / "full.parquet")).to_frame()
clean_export = result.export("clean", str(EXPORT_DIR / "clean.parquet")).to_frame()
dropped_export = result.export("dropped", str(EXPORT_DIR / "dropped.parquet")).to_frame()

action_counts_map = evaluation_table["final_action"].value_counts(dropna=False).to_dict()
expected_clean_count = int(action_counts_map.get("keep", 0))
expected_dropped_count = int(action_counts_map.get("drop", 0))

if len(full_export) != len(evaluation_table):
    raise AssertionError("full export row count mismatch")
if len(clean_export) != expected_clean_count:
    raise AssertionError("clean export row count mismatch")
if len(dropped_export) != expected_dropped_count:
    raise AssertionError("dropped export row count mismatch")

print("full_export_rows:", len(full_export))
print("clean_export_rows:", len(clean_export))
print("dropped_export_rows:", len(dropped_export))
print("PASS: cleaning v3 planner/scheduler smoke validation completed")
